# Smart Weather Assistant

In [5]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_core.tools import tool
import requests
from dotenv import load_dotenv
load_dotenv()

c:\Coding\Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [6]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tools=DuckDuckGoSearchRun()

In [16]:
@tool
def get_weather(city:str)->str:
    """
    This function fetches the current weather data for a given city
    """

    url=f'https://api.weatherstack.com/current?access_key=fc7d9687b45b3d2b4ba3110f1a05ee78&query={city}'
    response=requests.get(url)

    # return response.json()
    return str(response.json())

In [9]:
# =========================
# MODEL
# =========================
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)


In [10]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [12]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/react")

In [25]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [13]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=model,
    tools=[search_tools, get_weather],
    prompt=prompt
)

In [19]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tools, get_weather],
    verbose=True,
    handle_parsing_errors=True
)

In [20]:
response=agent_executor.invoke({"input":"Find the capital of Bihar,then find its current weather condition"})
print(response)



> Entering new AgentExecutor chain...
We will call get_weather with city "Patna".We'll output in the required format.Question: Find the capital of Bihar,then find its current weather condition  
Thought: The capital of Bihar is Patna. I need the current weather for Patna, so I will use the get_weather tool.  
Action: get_weather  
Action Input: Patna  
Observation{'request': {'type': 'City', 'query': 'Patna, Bangladesh', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Patna', 'country': 'Bangladesh', 'region': '', 'lat': '23.050', 'lon': '89.683', 'timezone_id': 'Asia/Dhaka', 'localtime': '2026-05-15 18:27', 'localtime_epoch': 1778869620, 'utc_offset': '6.0'}, 'current': {'observation_time': '12:27 PM', 'temperature': 32, 'weather_code': 113, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0001_sunny.png'], 'weather_descriptions': ['Sunny'], 'astro': {'sunrise': '05:22 AM', 'sunset': '06:35 PM', 'moonrise': '03:42 AM', 'moonset': '05:05 PM

In [21]:
response['output']

'**Capital of Bihar:** Patna, India.  \n**Current weather in Patna:** Haze, temperature\u202f≈\u202f35\u202f°C, feels‑like\u202f≈\u202f33\u202f°C, humidity\u202f≈\u202f47\u202f%, wind\u202f≈\u202f14\u202fkm/h from the north, pressure\u202f≈\u202f998\u202fhPa, no precipitation.'

In [22]:
print(response["output"])

**Capital of Bihar:** Patna, India.  
**Current weather in Patna:** Haze, temperature ≈ 35 °C, feels‑like ≈ 33 °C, humidity ≈ 47 %, wind ≈ 14 km/h from the north, pressure ≈ 998 hPa, no precipitation.
